In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import requests
import zipfile
import io
import csv
import re
import json
import time

# -- Declare widgets with defaults --------------------------------------
dbutils.widgets.text("catalog", "riskbricks")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")

# -- Read widgets -------------------------------------------------------
catalog = dbutils.widgets.get("catalog").strip()
local_tz = ZoneInfo("America/New_York")

# -- Ensure schemas exist -----------------------------------------------
spark.sql(f"USE CATALOG {catalog}")
for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

try:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
except Exception:
    pass

# -- Auto-detect date range ---------------------------------------------
start_input = dbutils.widgets.get("start_date").strip()
end_input = dbutils.widgets.get("end_date").strip()

if end_input:
    end_date = datetime.strptime(end_input, "%Y-%m-%d").replace(tzinfo=local_tz)
else:
    end_date = datetime.now(local_tz).replace(hour=0, minute=0, second=0, microsecond=0) - timedelta(days=1)

if start_input:
    start_date = datetime.strptime(start_input, "%Y-%m-%d").replace(tzinfo=local_tz)
else:
    # Auto-detect: MAX(event_date) + 1 day from existing table
    events_table = f"{catalog}.bronze.historical_news_gdelt"
    if spark.catalog.tableExists(events_table):
        max_dt = spark.sql(f"SELECT MAX(event_date) AS d FROM {events_table}").collect()[0].d
        if max_dt:
            start_date = datetime(max_dt.year, max_dt.month, max_dt.day, tzinfo=local_tz) + timedelta(days=1)
        else:
            start_date = end_date  # table empty, just do yesterday
    else:
        start_date = end_date  # table doesn't exist, just do yesterday

if start_date > end_date:
    msg = f"Nothing to ingest: start_date {start_date.date()} > end_date {end_date.date()}"
    print(msg)
    dbutils.notebook.exit(json.dumps({"status": "skipped", "message": msg}))

num_days = (end_date - start_date).days + 1
print(f"Date range: {start_date.date()} to {end_date.date()} ({num_days} days)")

In [0]:
# -- Load symbols from company_universe ---------------------------------
symbols_df = spark.sql(f"""
    SELECT DISTINCT symbol, company_name, sector
    FROM {catalog}.gold.company_universe
    ORDER BY symbol
""")

portfolio_symbols = [row.symbol for row in symbols_df.collect()]
symbol_to_company = {row.symbol: row.company_name for row in symbols_df.collect()}
symbol_to_sector = {row.symbol: row.sector for row in symbols_df.collect()}

# -- Build keyword map dynamically --------------------------------------
# Known short-name overrides (additive to dynamic approach)
COMPANY_KEYWORDS = {
    "AAPL": ["APPLE"], "MSFT": ["MICROSOFT"], "GOOGL": ["GOOGLE", "ALPHABET"],
    "AMZN": ["AMAZON"], "TSLA": ["TESLA"], "NVDA": ["NVIDIA"],
    "META": ["FACEBOOK"], "NFLX": ["NETFLIX"], "COST": ["COSTCO"],
    "JPM": ["JPMORGAN"], "BAC": ["BANK AMERICA"], "GS": ["GOLDMAN SACHS"],
    "MS": ["MORGAN STANLEY"], "WMT": ["WALMART"], "HD": ["HOME DEPOT"],
}

keyword_map = {}
for symbol in portfolio_symbols:
    keywords = set()
    sym_upper = (symbol or "").upper()
    if sym_upper:
        keywords.add(sym_upper)
    # Add manual overrides
    if symbol in COMPANY_KEYWORDS:
        keywords.update([kw.upper() for kw in COMPANY_KEYWORDS[symbol] if kw])
    # Dynamic: split company_name, keep tokens >= 4 chars
    company_name = symbol_to_company.get(symbol, symbol) or ""
    tokens = re.split(r"\s+", company_name.upper())
    for token in tokens:
        token = token.replace(".", "").replace(",", "").replace("'S", "")
        token = token.replace("INC", "").replace("CORP", "").replace("LLC", "").strip()
        if len(token) >= 4:
            keywords.add(token)
    keyword_map[symbol] = sorted([kw for kw in keywords if kw])

print(f"Loaded {len(portfolio_symbols)} symbols, keyword map built")

In [0]:
def _date_range(start_dt, end_dt):
    current = start_dt
    while current <= end_dt:
        yield current
        current += timedelta(days=1)

def _match_symbols(text_blob, keyword_map):
    matches = []
    text_upper = (text_blob or "").upper()
    for symbol, keywords in keyword_map.items():
        for kw in keywords:
            if len(kw) >= 4 and kw in text_upper:
                matches.append(symbol)
                break
    return matches

def _http_get_with_retry(url, retries=3, backoff=2, timeout=60):
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=timeout)
            return resp
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            if attempt < retries - 1:
                wait = backoff * (2 ** attempt)
                print(f"  Retry {attempt+1}/{retries} in {wait}s: {e}")
                time.sleep(wait)
            else:
                raise
    return None

def process_event_day(date_str, url, keyword_map):
    rows = []
    try:
        resp = _http_get_with_retry(url)
        if resp.status_code == 404:
            print(f"  {date_str}: No data (weekend/holiday)")
            return rows
        if resp.status_code != 200:
            print(f"  {date_str}: HTTP {resp.status_code}")
            return rows

        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            name = zf.namelist()[0]
            with zf.open(name) as f:
                reader = csv.reader(
                    io.TextIOWrapper(f, encoding="utf-8", errors="ignore"),
                    delimiter="\t",
                )
                for row in reader:
                    if len(row) < 35:
                        continue
                    try:
                        event_id = row[0]
                        event_date = row[1]
                        actor1_name = row[6]
                        actor2_name = row[16]
                        goldstein_scale = float(row[30]) if row[30] else 0.0
                        num_mentions = int(row[31]) if row[31] else 0
                        num_sources = int(row[32]) if row[32] else 0
                        num_articles = int(row[33]) if row[33] else 0
                        avg_tone = float(row[34]) if row[34] else 0.0
                        source_url = row[60] if len(row) > 60 else None

                        text_blob = f"{actor1_name} {actor2_name}"
                        matched_symbols = _match_symbols(text_blob, keyword_map)
                        if not matched_symbols:
                            continue

                        for symbol in matched_symbols:
                            rows.append({
                                "event_id": str(event_id),
                                "event_date": event_date,
                                "symbol": symbol,
                                "company_name": symbol_to_company.get(symbol, symbol),
                                "sector": symbol_to_sector.get(symbol, "Unknown"),
                                "actor1_name": actor1_name,
                                "actor2_name": actor2_name,
                                "goldstein_scale": goldstein_scale,
                                "num_mentions": num_mentions,
                                "num_sources": num_sources,
                                "num_articles": num_articles,
                                "avg_tone": avg_tone,
                                "source_url": source_url,
                                "source_file_date": date_str,
                                "ingestion_timestamp": datetime.now(local_tz),
                            })
                    except Exception:
                        continue
    except Exception as exc:
        print(f"  {date_str}: ERROR {exc}")
        return rows

    print(f"  {date_str}: {len(rows)} events")
    return rows

# -- Download and filter events -----------------------------------------
print(f"Downloading GDELT Events for {num_days} days...")
all_events = []
for day in _date_range(start_date, end_date):
    date_str = day.strftime("%Y%m%d")
    url = f"http://data.gdeltproject.org/events/{date_str}.export.CSV.zip"
    all_events.extend(process_event_day(date_str, url, keyword_map))

print(f"\nTotal events collected: {len(all_events)}")

In [0]:
def write_partitioned_table(table_name, df, start_dt, end_dt, partition_cols=("event_date", "symbol")):
    """Idempotent write using replaceWhere for the date range."""
    partition_col = partition_cols[0]
    replace_where = (
        f"{partition_col} >= '{start_dt.strftime('%Y-%m-%d')}' AND "
        f"{partition_col} <= '{end_dt.strftime('%Y-%m-%d')}'"
    )
    df = df.filter(
        (F.col(partition_col) >= F.lit(start_dt.strftime("%Y-%m-%d")).cast("date"))
        & (F.col(partition_col) <= F.lit(end_dt.strftime("%Y-%m-%d")).cast("date"))
    )
    if not spark.catalog.tableExists(table_name):
        df.write.mode("overwrite").partitionBy(*partition_cols).option(
            "overwriteSchema", "true"
        ).saveAsTable(table_name)
    else:
        df.write.mode("overwrite").option("replaceWhere", replace_where).saveAsTable(
            table_name
        )

# -- Events schema and write --------------------------------------------
events_written = 0
if all_events:
    events_schema = StructType([
        StructField("event_id", StringType(), True),
        StructField("event_date", StringType(), True),
        StructField("symbol", StringType(), True),
        StructField("company_name", StringType(), True),
        StructField("sector", StringType(), True),
        StructField("actor1_name", StringType(), True),
        StructField("actor2_name", StringType(), True),
        StructField("goldstein_scale", DoubleType(), True),
        StructField("num_mentions", IntegerType(), True),
        StructField("num_sources", IntegerType(), True),
        StructField("num_articles", IntegerType(), True),
        StructField("avg_tone", DoubleType(), True),
        StructField("source_url", StringType(), True),
        StructField("source_file_date", StringType(), True),
        StructField("ingestion_timestamp", TimestampType(), True),
    ])
    events_df = spark.createDataFrame(all_events, schema=events_schema)
    events_df = events_df.withColumn(
        "event_date",
        F.coalesce(
            F.expr("try_to_date(CAST(event_date AS STRING), 'yyyyMMdd')"),
            F.expr("try_to_date(CAST(source_file_date AS STRING), 'yyyyMMdd')")
        ).cast("date"),
    )
    events_table = f"{catalog}.bronze.historical_news_gdelt"
    write_partitioned_table(events_table, events_df, start_date, end_date)
    events_written = events_df.count()
    print(f"Saved {events_written} events to {events_table}")
else:
    print("No events to write")